## Membangun Katalog

In [2]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db_kopi = client["kopi_nusantara"]
koleksi_menu = db_kopi["menu"]

# Hapus data lama supaya bisa dijalankan ulang
koleksi_menu.delete_many({})

koleksi_menu.insert_many([
    # Menu Kopi (punya tingkat_kepahitan)
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 50},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 45},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 60},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 40},
    {"nama_menu": "Vietnam Drip", "harga": 24000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 30},
    {"nama_menu": "Kopi Tubruk", "harga": 12000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 55},

    # Menu Makanan (punya pedas_level)
    {"nama_menu": "Nasi Goreng Kampung", "harga": 28000, "kategori": "Makanan", "pedas_level": 2, "stok": 25},
    {"nama_menu": "Mie Goreng Pedas", "harga": 25000, "kategori": "Makanan", "pedas_level": 3, "stok": 30},
    {"nama_menu": "Roti Bakar Coklat", "harga": 18000, "kategori": "Makanan", "pedas_level": 0, "stok": 35},
    {"nama_menu": "Kentang Goreng", "harga": 14000, "kategori": "Makanan", "pedas_level": 0, "stok": 40},
    {"nama_menu": "Ayam Geprek Mini", "harga": 32000, "kategori": "Makanan", "pedas_level": 3, "stok": 20},
    {"nama_menu": "Pisang Goreng Keju", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 45},
])

print("Jumlah menu:", koleksi_menu.count_documents({}))

# jumlah menu harus 12

Jumlah menu: 12


## Kueri Bertingkat (Read)

In [3]:
# B1: menu Kopi dengan tingkat_kepahitan di atas 3
print("Kopi dengan tingkat kepahitan > 3:")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {m['nama_menu']} (kepahitan {m['tingkat_kepahitan']})")

# B2: menu dengan harga di antara 15000 dan 30000
print("\nMenu dengan harga 15000 sampai 30000:")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  - {m['nama_menu']} (Rp{m['harga']:,})")

# B3: menu yang tidak punya field pedas_level
print("\nMenu tanpa field pedas_level:")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {m['nama_menu']}")

Kopi dengan tingkat kepahitan > 3:
  - Espresso (kepahitan 5)
  - Americano (kepahitan 4)
  - Vietnam Drip (kepahitan 4)
  - Kopi Tubruk (kepahitan 5)

Menu dengan harga 15000 sampai 30000:
  - Espresso (Rp18,000)
  - Americano (Rp20,000)
  - Kopi Susu Gula Aren (Rp22,000)
  - Cappuccino (Rp25,000)
  - Vietnam Drip (Rp24,000)
  - Nasi Goreng Kampung (Rp28,000)
  - Mie Goreng Pedas (Rp25,000)
  - Roti Bakar Coklat (Rp18,000)
  - Pisang Goreng Keju (Rp15,000)

Menu tanpa field pedas_level:
  - Espresso
  - Americano
  - Kopi Susu Gula Aren
  - Cappuccino
  - Vietnam Drip
  - Kopi Tubruk


## Pembaruan Data (Updatae)

In [4]:
# Semua menu diberi status Tersedia
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})

# Dua menu diubah menjadi Habis
koleksi_menu.update_one({"nama_menu": "Kopi Tubruk"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Ayam Geprek Mini"}, {"$set": {"status": "Habis"}})

for m in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {m['nama_menu']}: {m['status']}")

  - Espresso: Tersedia
  - Americano: Tersedia
  - Kopi Susu Gula Aren: Tersedia
  - Cappuccino: Tersedia
  - Vietnam Drip: Tersedia
  - Kopi Tubruk: Habis
  - Nasi Goreng Kampung: Tersedia
  - Mie Goreng Pedas: Tersedia
  - Roti Bakar Coklat: Tersedia
  - Kentang Goreng: Tersedia
  - Ayam Geprek Mini: Habis
  - Pisang Goreng Keju: Tersedia


## Integrasi ke spark

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import avg

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/05 00:10:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# Skema dibuat manual karena tidak semua dokumen punya field yang sama
skema = StructType([
    StructField("_id", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("harga", IntegerType(), True),
    StructField("kategori", StringType(), True),
    StructField("tingkat_kepahitan", IntegerType(), True),
    StructField("pedas_level", IntegerType(), True),
    StructField("stok", IntegerType(), True),
    StructField("status", StringType(), True),
])

# Ambil dokumen, field yang tidak ada diisi None
baris = []
for m in koleksi_menu.find({}):
    baris.append((
        str(m["_id"]),
        m["nama_menu"],
        m["harga"],
        m["kategori"],
        m.get("tingkat_kepahitan"),
        m.get("pedas_level"),
        m["stok"],
        m["status"],
    ))

df_menu = spark.createDataFrame(baris, skema)

# 1. Skema hasil konversi
df_menu.printSchema()

# Tabel lengkap, untuk melihat nilai null
df_menu.show()

# 2. Rata-rata harga per kategori
df_menu.groupBy("kategori").agg(avg("harga").alias("rata_rata_harga")).show()

spark.stop()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: integer (nullable = true)
 |-- pedas_level: integer (nullable = true)
 |-- stok: integer (nullable = true)
 |-- status: string (nullable = true)



+--------------------+-------------------+-----+--------+-----------------+-----------+----+--------+
|                 _id|          nama_menu|harga|kategori|tingkat_kepahitan|pedas_level|stok|  status|
+--------------------+-------------------+-----+--------+-----------------+-----------+----+--------+
|6aeb662b27d9db123...|           Espresso|18000|    Kopi|                5|       NULL|  50|Tersedia|
|6aeb662b27d9db123...|          Americano|20000|    Kopi|                4|       NULL|  45|Tersedia|
|6aeb662b27d9db123...|Kopi Susu Gula Aren|22000|    Kopi|                2|       NULL|  60|Tersedia|
|6aeb662b27d9db123...|         Cappuccino|25000|    Kopi|                3|       NULL|  40|Tersedia|
|6aeb662b27d9db123...|       Vietnam Drip|24000|    Kopi|                4|       NULL|  30|Tersedia|
|6aeb662b27d9db123...|        Kopi Tubruk|12000|    Kopi|                5|       NULL|  55|   Habis|
|6aeb662b27d9db123...|Nasi Goreng Kampung|28000| Makanan|             NULL|       

+--------+------------------+
|kategori|   rata_rata_harga|
+--------+------------------+
|    Kopi|20166.666666666668|
| Makanan|           22000.0|
+--------+------------------+

